# EBC: behavior matching and HD-only controls

All active good units are analyzed; RF/HD labels only group the results. No GLM is used.

Reference: [Peyrache, Schieferstein & Buzsáki (2017), Transformation of the head-direction signal into a spatial code](https://doi.org/10.1038/s41467-017-01908-3), Fig. 1–3 and Methods. The HD-only Poisson control follows the paper; held-out time folds, full d(θ) matching, and split/position correlations are additional controls.


In [ ]:
%matplotlib inline
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import json
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.sparse import csr_matrix
from scipy.spatial import cKDTree
from scipy.stats import false_discovery_control
from Utils.ebc_analysis import (
    load_basler_session, rectangle_distances, ray_histogram_matrix,
    rates_from_counts, band_mask, row_spearman,
)
from Utils.json_tools import read_formatted_json
from Utils.plotting import apply_light_plot_style

apply_light_plot_style()


In [ ]:
root = Path("/mnt/senzailab/Kai/#Recording/m19")
date = "260831"
session = root / date / f"{date}_2"
probe = "A"
unit_id = 267  # Re-run only the plotting cells to inspect another unit.
simulations = 500
bootstraps = 1000
workers = 4
seed = 0
block_s = 10.
geometry_rms_cm = 2.
position_separation_cm = 10.
hd_separation_deg = 60.
save_figures = False
output_dir = session / "data/spatial_cells/ebc_artifact_controls"
output_dir.mkdir(parents=True, exist_ok=True)

def show(figure, name):
    if save_figures:
        figure.savefig(output_dir / f"{name}.png", dpi=180, facecolor="white", transparent=False)
    display(figure)
    plt.close(figure)


## Analysis functions

Data preparation, statistical controls, and plotting are defined here. Run these definition cells before the analysis below.


In [ ]:
BANDS12 = (("0-8", None, 8), ("8-16", 8, 16), ("16-", 16, None))
THETA_DEG = np.arange(0., 360., 6.)
BASLER_BOUNDS_PX = (370., 920., 210., 760.)  # left, right, top, bottom
BASLER_SIZE_CM = 41.


In [ ]:
def prepare_controls(data):
    valid = np.all((data["xy"] >= 0) & (data["xy"] <= 41), axis=1)
    times, xy, hd = data["times"][valid], data["xy"][valid], data["hd"][valid]
    counts = data["counts"][:, valid]
    active = counts.sum(axis=1) > 0
    smooth_xy = gaussian_filter1d(data["xy"], .1 / data["dt"], axis=0)
    speed = np.linalg.norm(np.gradient(smooth_xy, data["times"], axis=0), axis=1)[valid]
    distances = rectangle_distances(xy, hd, theta_deg=THETA_DEG, size_cm=BASLER_SIZE_CM)
    edges = np.linspace(0, 41 / 2 * np.sqrt(2), 21)
    projection = ray_histogram_matrix(distances, edges)
    spatial_bin = np.floor(xy / 2).astype(int)
    spatial_index = spatial_bin[:, 1] * 21 + spatial_bin[:, 0]
    position_projection = csr_matrix((np.ones(len(times)), (spatial_index, np.arange(len(times)))), shape=(441, len(times)))
    return dict(times=times, xy=xy, hd=hd, speed=speed, distances=distances,
                counts=counts[active], unit_ids=data["unit_ids"][active], dt=data["dt"],
                edges=edges, projection=projection, position_projection=position_projection,
                excluded_geometry_frames=int((~valid).sum()), shape=(60, 20))


def behavior_states(data, *, speed_stratified=False):
    keys = np.c_[np.floor(data["xy"] / 2).astype(int), np.floor(data["hd"] / 12).astype(int)]
    if speed_stratified:
        keys = np.c_[keys, np.digitize(data["speed"], [2.5, 5, 10])]
    states, frame_state, n = np.unique(keys, axis=0, return_inverse=True, return_counts=True)
    projection = csr_matrix((np.ones(len(frame_state)), (frame_state, np.arange(len(frame_state)))), shape=(len(states), len(frame_state)))
    xy = (projection @ data["xy"]) / n[:, None]
    direction = np.rad2deg(np.angle(projection @ np.exp(1j * np.deg2rad(data["hd"])))) % 360
    geometry = (projection @ data["distances"]) / n[:, None]
    return dict(keys=states, frame_state=frame_state, xy=xy, hd=direction, geometry=geometry,
                occupancy=n * data["dt"], counts=(projection @ data["counts"].T).T,
                speed_stratified=speed_stratified)


def match_states(states, *, mode, rms_cm=2., separation_cm=10., hd_separation_deg=60.):
    geometry, xy, hd = states["geometry"], states["xy"], states["hd"]
    if mode == "geometry":
        pairs = cKDTree(geometry).query_pairs(rms_cm * np.sqrt(geometry.shape[1]), output_type="ndarray")
    else:
        groups = states["keys"][:, 2:]
        _, group_ids = np.unique(groups, axis=0, return_inverse=True)
        pieces = []
        for group in np.unique(group_ids):
            indices = np.flatnonzero(group_ids == group)
            a, b = np.triu_indices(len(indices), 1)
            pieces.append(np.c_[indices[a], indices[b]])
        pairs = np.concatenate(pieces)
    a, b = pairs.T
    separation = np.linalg.norm(xy[a] - xy[b], axis=1)
    hd_delta = np.abs((hd[a] - hd[b] + 180) % 360 - 180)
    rms = np.sqrt(np.mean((geometry[a] - geometry[b])**2, axis=1))
    selected = separation >= separation_cm
    if mode == "geometry":
        selected &= hd_delta >= hd_separation_deg
    common_s = np.minimum(states["occupancy"][a], states["occupancy"][b])
    score = rms if mode == "geometry" else -common_s
    used, rows = set(), []
    for i in np.flatnonzero(selected)[np.argsort(score[selected], kind="stable")]:
        first, second = map(int, pairs[i])
        if first in used or second in used:
            continue
        used.update((first, second))
        rows.append((first, second, rms[i], separation[i], hd_delta[i], common_s[i]))
    return pd.DataFrame(rows, columns=["state_a", "state_b", "geometry_rms_cm", "position_separation_cm", "hd_separation_deg", "matched_seconds"])


def matched_firing_rates(data, states, pairs, *, bootstraps=1000, block_s=10., seed=0):
    rng = np.random.default_rng(seed)
    blocks = ((data["times"] - data["times"][0]) / block_s).astype(int)
    n_blocks = blocks.max() + 1
    block_map = csr_matrix((np.ones(len(blocks)), (blocks, np.arange(len(blocks)))), shape=(n_blocks, len(blocks)))
    draws = rng.multinomial(n_blocks, np.full(n_blocks, 1 / n_blocks), size=bootstraps)
    rates, bootstrap_rates, exposures, spikes = [], [], [], []
    for side in ("state_a", "state_b"):
        weights = np.zeros(len(states["occupancy"]))
        selected = pairs[side].to_numpy(int)
        weights[selected] = pairs.matched_seconds.to_numpy() / states["occupancy"][selected]
        frame_weights = weights[states["frame_state"]]
        exposure = np.asarray(block_map @ frame_weights).ravel() * data["dt"]
        block_spikes = block_map @ (data["counts"] * frame_weights).T
        total_spikes = block_spikes.sum(axis=0)
        rates.append(np.divide(total_spikes, exposure.sum(), out=np.full(len(total_spikes), np.nan), where=exposure.sum() > 0))
        sampled_time = draws @ exposure
        bootstrap_rates.append(np.divide(draws @ block_spikes, sampled_time[:, None],
                                        out=np.full((bootstraps, len(total_spikes)), np.nan), where=sampled_time[:, None] > 0))
        exposures.append(exposure)
        spikes.append(total_spikes)
    differences = bootstrap_rates[1] - bootstrap_rates[0]
    ci = np.array([np.quantile(v[np.isfinite(v)], [.025, .975]) if np.isfinite(v).any() else [np.nan, np.nan] for v in differences.T])
    return pd.DataFrame({"unit_id": data["unit_ids"], "rate_a_hz": rates[0], "rate_b_hz": rates[1],
                         "difference_hz": rates[1] - rates[0], "ci_low": ci[:, 0], "ci_high": ci[:, 1],
                         "matched_seconds": pairs.matched_seconds.sum(), "pair_count": len(pairs),
                         "occupied_time_blocks": np.count_nonzero((exposures[0] > 0) | (exposures[1] > 0)),
                         "weighted_spikes_a": spikes[0], "weighted_spikes_b": spikes[1]}).set_index("unit_id")


def hd_expected_counts(data, *, folds=5, bins=60, smoothing_deg=3., guard_s=5.):
    """Train each HD lookup outside its test time fold and a temporal guard."""
    times, dt = data["times"], data["dt"]
    edges = np.linspace(times[0], times[-1] + dt, folds + 1)
    hd_bin = np.floor(data["hd"] * bins / 360).astype(int)
    lookup_map = csr_matrix((np.ones(len(times)), (hd_bin, np.arange(len(times)))), shape=(bins, len(times)))
    expected = np.full(data["counts"].shape, np.nan)
    for start, end in zip(edges[:-1], edges[1:]):
        test = (times >= start) & (times < end)
        train = (times < start - guard_s) | (times >= end + guard_s)
        occupancy = gaussian_filter1d(np.asarray(lookup_map @ train.astype(float)).ravel() * dt, smoothing_deg / (360 / bins), mode="wrap")
        counts = (lookup_map @ (data["counts"] * train).T).T
        counts = gaussian_filter1d(counts, smoothing_deg / (360 / bins), axis=1, mode="wrap")
        rates = np.divide(counts, occupancy, out=np.full_like(counts, np.nan), where=occupancy > 0)
        expected[:, test] = rates[:, hd_bin[test]] * dt
    return expected


def angular_profiles(maps, edges):
    centers = (edges[:-1] + edges[1:]) / 2
    curves = []
    for _, lower, upper in BANDS12:
        values = maps[:, :, band_mask(centers, lower, upper)]
        curve = np.nansum(values, axis=2)
        curve[~np.isfinite(values).any(axis=2)] = np.nan
        curves.append(curve)
    return np.stack(curves, axis=1)


def tuning_strength(curves):
    vector = np.nansum(curves * np.exp(1j * np.deg2rad(THETA_DEG + 3)), axis=-1)
    total = np.nansum(curves, axis=-1)
    return np.divide(np.abs(vector), total, out=np.full(total.shape, np.nan), where=total > 0)


def spatial_information(counts, occupancy):
    smoothed_counts = gaussian_filter(counts.reshape(-1, 21, 21).astype(float), (0, 1, 1), mode="nearest")
    smoothed_occupancy = gaussian_filter(occupancy.reshape(21, 21), 1, mode="nearest")
    maps = np.divide(smoothed_counts, smoothed_occupancy, out=np.full_like(smoothed_counts, np.nan), where=smoothed_occupancy > 0)
    # Normalize the smoothed map using the same occupancy measure as the
    # information integral; a raw-count mean can produce negative information.
    mean_rate = np.nansum(maps * occupancy.reshape(21, 21), axis=(1, 2)) / occupancy.sum()
    relative = np.divide(maps, mean_rate[:, None, None], out=np.zeros_like(maps), where=mean_rate[:, None, None] > 0)
    log = np.log2(relative, out=np.zeros_like(relative), where=relative > 0)
    return np.nansum(relative * log * occupancy.reshape(21, 21) / occupancy.sum(), axis=(1, 2))


def hd_only_controls(data, *, simulations=500, batch_size=25, workers=4, seed=0, sigma=5):
    expected = hd_expected_counts(data)
    support = np.isfinite(expected).all(axis=0)
    expected[:, ~support] = 0
    projection = data["projection"]
    position_projection = data["position_projection"]
    occupancy = (projection @ (support * data["dt"])).reshape(data["shape"])
    spatial_occ = position_projection @ (support * data["dt"])
    observed_counts = data["counts"] * support
    observed = rates_from_counts((projection @ observed_counts.T).T.reshape(-1, *data["shape"]), occupancy, sigma=sigma)
    predicted = rates_from_counts((projection @ expected.T).T.reshape(-1, *data["shape"]), occupancy, sigma=sigma)
    curves = angular_profiles(observed, data["edges"])
    strength = tuning_strength(curves)
    information = spatial_information((position_projection @ observed_counts.T).T, spatial_occ)
    seeds = np.random.SeedSequence(seed).spawn(len(data["unit_ids"]))

    def simulate(index):
        rng = np.random.default_rng(seeds[index])
        all_curves = np.empty((simulations, 3, len(THETA_DEG)), dtype=np.float32)
        all_information = np.empty(simulations)
        for start in range(0, simulations, batch_size):
            end = min(start + batch_size, simulations)
            spikes = rng.poisson(expected[index], size=(end - start, expected.shape[1])).astype(np.int32)
            maps = rates_from_counts((projection @ spikes.T).T.reshape(-1, *data["shape"]), occupancy, sigma=sigma)
            all_curves[start:end] = angular_profiles(maps, data["edges"])
            all_information[start:end] = spatial_information((position_projection @ spikes.T).T, spatial_occ)
        bounds = np.nanquantile(all_curves, [.025, .975], axis=0)
        return tuning_strength(all_curves), all_information, bounds

    with ThreadPoolExecutor(max_workers=workers) as pool:
        results = []
        for i, result in enumerate(pool.map(simulate, range(len(seeds)))):
            results.append(result)
            if (i + 1) % 10 == 0 or i + 1 == len(seeds):
                print(f"HD-only controls: {i + 1}/{len(seeds)} units, {simulations} simulations each", flush=True)
    null_strength = np.stack([r[0] for r in results])
    null_info = np.stack([r[1] for r in results])
    bounds = np.stack([r[2] for r in results])
    defined_null = np.isfinite(null_strength).sum(axis=1)
    p = (1 + np.sum(null_strength >= strength[:, None, :], axis=1)) / (defined_null + 1)
    p[~np.isfinite(strength)] = np.nan
    q = np.full_like(p, np.nan)
    finite = np.isfinite(p)
    q[finite] = false_discovery_control(p[finite])
    table = pd.DataFrame(index=pd.Index(data["unit_ids"], name="unit_id"))
    table["spikes"] = observed_counts.sum(axis=1)
    table["mean_rate_hz"] = table.spikes / (support.sum() * data["dt"])
    table["hd_prediction_coverage"] = support.mean()
    table["spatial_info"] = information
    table["hd_null_spatial_info"] = null_info.mean(axis=1)
    table["unbiased_spatial_info"] = information - null_info.mean(axis=1)
    table["spatial_info_p"] = (1 + np.sum(null_info >= information[:, None], axis=1)) / (simulations + 1)
    for i, (band, _, _) in enumerate(BANDS12):
        table[f"mvl_{band}"] = strength[:, i]
        table[f"hd_null_mvl_{band}"] = np.nanmean(null_strength[:, :, i], axis=1)
        table[f"defined_null_samples_{band}"] = defined_null[:, i]
        table[f"p_{band}"] = p[:, i]
        table[f"q_{band}"] = q[:, i]
    return dict(table=table, observed=observed, predicted=predicted, curves=curves,
                expected_curves=angular_profiles(predicted, data["edges"]), null_curve_bounds=bounds,
                null_strength=null_strength, null_spatial_information=null_info, occupancy=occupancy,
                expected_counts=expected, support=support)


def curve_correlations(first, second):
    correlations = np.full(first.shape[:2], np.nan)
    for unit in range(len(first)):
        for band in range(first.shape[1]):
            valid = np.isfinite(first[unit, band]) & np.isfinite(second[unit, band])
            if valid.sum() >= 3:
                correlations[unit, band] = row_spearman(first[unit, band, valid], second[unit, band, valid])
    return correlations


def stability_controls(data):
    def maps(selected, sigma=5):
        occ = (data["projection"] @ (selected * data["dt"])).reshape(data["shape"])
        counts = (data["projection"] @ (data["counts"] * selected).T).T.reshape(-1, *data["shape"])
        return rates_from_counts(counts, occ, sigma=sigma)

    def compare(first, second):
        # Both curves project exactly the same angle/distance support.
        common = np.isfinite(first) & np.isfinite(second)
        return curve_correlations(angular_profiles(np.where(common, first, np.nan), data["edges"]),
                                  angular_profiles(np.where(common, second, np.nan), data["edges"]))

    all_frames = np.ones(len(data["times"]), dtype=bool)
    early = data["times"] < np.mean([data["times"][0], data["times"][-1]])
    left = data["xy"][:, 0] < 20.5
    full = maps(all_frames)
    tests = {
        "half": compare(maps(early), maps(~early)),
        "position": compare(maps(left), maps(~left)),
        "moving": compare(full, maps(data["speed"] > 2.5)),
        "light_smoothing": compare(full, maps(all_frames, sigma=1)),
    }
    table = pd.DataFrame(index=pd.Index(data["unit_ids"], name="unit_id"))
    for name, values in tests.items():
        for i, (band, _, _) in enumerate(BANDS12):
            table[f"{name}_rho_{band}"] = values[:, i]
    return table


def evidence_table(controls, stability, geometry_rates, hd_rates):
    table = controls["table"].join(stability)
    table = table.join(geometry_rates.add_prefix("same_geometry_")).join(hd_rates.add_prefix("same_hd_"))
    significant = np.column_stack([table[f"q_{b}"] < .05 for b, _, _ in BANDS12])
    stable = np.column_stack([(table[f"half_rho_{b}"] >= .5) & (table[f"position_rho_{b}"] >= .5)
                              for b, _, _ in BANDS12])
    support = (table.spikes >= 50) & (table.same_geometry_matched_seconds >= 10) & (table.same_hd_matched_seconds >= 10) & (table.hd_prediction_coverage >= .95)
    weighted_spikes = table[[f"{comparison}_weighted_spikes_{side}"
                             for comparison in ("same_geometry", "same_hd") for side in ("a", "b")]]
    support &= weighted_spikes.min(axis=1) >= 20
    supported_effect = np.any(significant & stable, axis=1)
    table["conclusion"] = "HD/采样偏差可解释"
    table.loc[supported_effect & support, "conclusion"] = "仍有额外 EBC 证据"
    table.loc[~support | (significant.any(axis=1) & ~supported_effect), "conclusion"] = "采样不足"
    table["sampling_supported"] = support
    table["stable_across_time_and_position"] = stable.any(axis=1)
    table["additional_ebc_bands"] = [", ".join(b for (b, _, _), valid in zip(BANDS12, row) if valid)
                                      for row in significant & stable]
    return table




In [ ]:
def plot_matched_conditions(data, states, pairs, rates, unit_id, *, title):
    unit = np.flatnonzero(data["unit_ids"] == unit_id)[0]
    fig, axes = plt.subplots(1, 4, figsize=(19, 4), layout="constrained")
    axes[0].plot(*data["xy"][::10].T, color=".85", lw=.4)
    for side, color in (("state_a", "tab:blue"), ("state_b", "tab:orange")):
        ids = pairs[side].to_numpy(int)
        axes[0].scatter(*states["xy"][ids].T, s=8, color=color, label=side[-1].upper())
    axes[0].set(xlim=(0, 41), ylim=(0, 41), xlabel="X (cm)", ylabel="Y (cm)", title=f"{title}\n{len(pairs)} behavior-matched pairs")
    axes[0].set_aspect("equal")
    axes[0].legend()
    for i, pair in enumerate(pairs.nlargest(3, "matched_seconds").itertuples()):
        color = f"C{i}"
        axes[1].plot(THETA_DEG, states["geometry"][pair.state_a], color=color, label=f"Pair {i+1} A")
        axes[1].plot(THETA_DEG, states["geometry"][pair.state_b], color=color, ls="--", label=f"Pair {i+1} B")
    axes[1].set(xlabel="Egocentric bearing (°)", ylabel="d(θ) (cm)", title="Examples with highest common occupancy")
    axes[1].legend(fontsize=7)
    a, b = pairs.state_a.to_numpy(int), pairs.state_b.to_numpy(int)
    occupancy_a, occupancy_b = states["occupancy"][a], states["occupancy"][b]
    axes[2].scatter(occupancy_a, occupancy_b, s=10, alpha=.4)
    limits = [min(occupancy_a.min(), occupancy_b.min()), max(occupancy_a.max(), occupancy_b.max())]
    axes[2].plot(limits, limits, color=".4", ls=":")
    axes[2].set(xscale="log", yscale="log", xlabel="A occupancy (s)", ylabel="B occupancy (s)",
                title="Raw state occupancy\nRates weighted to the common minimum")
    fa, fb = states["counts"][unit, a] / states["occupancy"][a], states["counts"][unit, b] / states["occupancy"][b]
    axes[3].scatter(fa, fb, s=8 + 20 * pairs.matched_seconds, alpha=.4)
    maximum = max(axes[3].get_xlim()[1], axes[3].get_ylim()[1])
    axes[3].plot([0, maximum], [0, maximum], color=".4", ls=":")
    row = rates.loc[unit_id]
    axes[3].set(xlabel="A firing rate (Hz)", ylabel="B firing rate (Hz)",
                title=f"Unit {unit_id} | matched {row.matched_seconds:.1f} s/side\nΔFR={row.difference_hz:.2f}, 95% CI [{row.ci_low:.2f}, {row.ci_high:.2f}]")
    return fig


def plot_unit_controls(data, controls, unit_id):
    i = np.flatnonzero(data["unit_ids"] == unit_id)[0]
    n_bands = len(BANDS12)
    ncols = max(3, n_bands)
    fig, axes = plt.subplots(2, ncols, figsize=(5 * ncols, 8), squeeze=False, layout="constrained")
    for ax in list(axes[0, 3:]) + list(axes[1, n_bands:]):
        ax.remove()
    observed, expected = controls["observed"][i], controls["predicted"][i]
    vmax = np.nanmax([observed, expected])
    for ax, values, title in zip(axes[0, :3], (observed, expected, observed - expected), ("Observed", "HD-only expected", "Observed minus expected"), strict=True):
        residual = title.startswith("Observed minus")
        scale = np.nanmax(np.abs(values)) if residual else vmax
        image = ax.pcolormesh(data["edges"], np.arange(0, 361, 6), values, shading="flat",
                              cmap="coolwarm" if residual else "viridis", vmin=-scale if residual else 0, vmax=scale)
        ax.set(xlabel="Boundary distance (cm)", ylabel="Bearing (°)", title=f"Unit {unit_id}: {title}")
        fig.colorbar(image, ax=ax, label="Hz")
    for b, (ax, (band, _, _)) in enumerate(zip(axes[1, :n_bands], BANDS12, strict=True)):
        ax.fill_between(THETA_DEG + 3, *controls["null_curve_bounds"][i, :, b], color=".85", label="HD-only 95% pointwise interval")
        ax.plot(THETA_DEG + 3, controls["curves"][i, b], label="Observed")
        ax.plot(THETA_DEG + 3, controls["expected_curves"][i, b], label="HD-only", color="tab:orange")
        ax.set(xlabel="Bearing (°)", ylabel="Sum of rate bins (Hz)", title=f"{band} cm | MRL q={controls['table'].loc[unit_id, f'q_{band}']:.3g}")
        ax.legend(fontsize=7)
    return fig


def plot_population_controls(table):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
    for band, _, _ in BANDS12:
        axes[0].scatter(table[f"hd_null_mvl_{band}"], table[f"mvl_{band}"], s=18, alpha=.65, label=f"{band} cm")
    axes[0].plot([0, 1], [0, 1], color=".4", ls=":")
    axes[0].set(xlabel="HD-only mean MRL", ylabel="Observed MRL", title="EBC modulation vs. behavioral confound")
    axes[0].legend(fontsize=8)
    axes[1].scatter(table.hd_null_spatial_info, table.spatial_info, s=20)
    maximum = max(axes[1].get_xlim()[1], axes[1].get_ylim()[1])
    axes[1].plot([0, maximum], [0, maximum], color=".4", ls=":")
    axes[1].set(xlabel="HD-only spatial information (bit/spike)", ylabel="Observed spatial information (bit/spike)")
    half = table[[f"half_rho_{b}" for b, _, _ in BANDS12]].median(axis=1)
    position = table[[f"position_rho_{b}" for b, _, _ in BANDS12]].median(axis=1)
    axes[2].scatter(half, position, s=20)
    axes[2].axhline(.5, color=".5", ls=":")
    axes[2].axvline(.5, color=".5", ls=":")
    axes[2].set(xlim=(-1, 1), ylim=(-1, 1), xlabel="Split-half Spearman ρ", ylabel="Across-position Spearman ρ", title="Median over distance bands")
    return fig


def plot_unit_null_distributions(data, controls, unit_id):
    i = np.flatnonzero(data["unit_ids"] == unit_id)[0]
    row = controls["table"].loc[unit_id]
    n_bands = len(BANDS12)
    samples = [controls["null_strength"][i, :, b] for b in range(n_bands)] + [controls["null_spatial_information"][i]]
    observed = [row[f"mvl_{b}"] for b, _, _ in BANDS12] + [row.spatial_info]
    titles = [f"{b} cm | MRL q={row[f'q_{b}']:.3g}" for b, _, _ in BANDS12]
    titles.append(f"Spatial information | p={row.spatial_info_p:.3g}")
    fig, axes = plt.subplots(1, n_bands + 1, figsize=(4.25 * (n_bands + 1), 3.5), squeeze=False, layout="constrained")
    for ax, null, value, title, label in zip(axes[0], samples, observed, titles, ["MRL"] * n_bands + ["bit/spike"], strict=True):
        ax.hist(null[np.isfinite(null)], bins=25, color=".8", label="HD-only Poisson")
        ax.axvline(value, color="tab:blue", lw=2, label="Observed")
        ax.set(xlabel=label, ylabel="Simulations", title=title)
        ax.legend(fontsize=7)
    fig.suptitle(f"Unit {unit_id}: HD-only null distributions")
    return fig


## Raw data and coverage

Valid camera frames use active-low exposure midpoints. Geometry uses a 41 cm square; the 27 pose samples outside its measured bounds are excluded from boundary analyses. Spikes are assigned to the nearest retained camera frame before this geometry mask is applied. Unvisited cells remain missing.


In [ ]:
raw = load_basler_session(session, bounds_px=BASLER_BOUNDS_PX, size_cm=BASLER_SIZE_CM, probe=probe)
data = prepare_controls(raw)

print({
    "active_good_units": len(data["unit_ids"]),
    "frames": len(data["times"]),
    "excluded_geometry_frames": data["excluded_geometry_frames"],
    "duration_s": len(data["times"]) * data["dt"],
})

print(raw["source"]["camera_timing"])

fig, ax = plt.subplots(figsize=(6, 5))

image = ax.hist2d(
    *data["xy"].T,
    bins=np.linspace(0, 41, 22),
    weights=np.full(len(data["times"]), data["dt"]),
    vmin=0,
    vmax=45,
)

ax.set(
    xlabel="X (cm)",
    ylabel="Y (cm)",
    title="Position occupancy",
)
ax.set_aspect("equal")

fig.colorbar(image[3], ax=ax, label="Seconds")

show(fig, "occupancy")

## 3.1 Similar full d(θ), different position and HD

Behavior states use 2 cm position bins and 12° HD bins. Match their mean 60-direction distance vectors within 2 cm RMS, with at least 10 cm position separation and 60° HD separation. Greedy matching uses no spikes and never reuses a state. Both sides receive the same common-occupancy weight. Differences use 10 s block bootstrap intervals; sparse individual-state rates are descriptive. Mixed HD×EBC tuning can retain an HD difference here.


In [ ]:
states = behavior_states(data)
geometry_pairs = match_states(states, mode="geometry", rms_cm=geometry_rms_cm,
                             separation_cm=position_separation_cm, hd_separation_deg=hd_separation_deg)
geometry_rates = matched_firing_rates(data, states, geometry_pairs, bootstraps=bootstraps, block_s=block_s, seed=seed)
geometry_pairs.to_csv(output_dir / "same_geometry_pairs.csv", index=False)
geometry_rates.to_csv(output_dir / "same_geometry_rates.csv")
display(geometry_rates)


In [ ]:
show(plot_matched_conditions(data, states, geometry_pairs, geometry_rates, unit_id, title="Similar d(θ), different position / HD"), f"unit_{unit_id}_same_geometry")

## 3.2 Similar HD, different positions

Use the same HD bin and positions at least 10 cm apart. Match the best shared occupancy first. The second comparison additionally matches speed bins: <2.5, 2.5–5, 5–10, and ≥10 cm/s.


In [ ]:
hd_pairs = match_states(states, mode="hd", separation_cm=position_separation_cm)
hd_rates = matched_firing_rates(data, states, hd_pairs, bootstraps=bootstraps, block_s=block_s, seed=seed)
speed_states = behavior_states(data, speed_stratified=True)
speed_pairs = match_states(speed_states, mode="hd", separation_cm=position_separation_cm)
speed_rates = matched_firing_rates(data, speed_states, speed_pairs, bootstraps=bootstraps, block_s=block_s, seed=seed)
hd_pairs.to_csv(output_dir / "same_hd_pairs.csv", index=False)
hd_rates.to_csv(output_dir / "same_hd_rates.csv")
speed_pairs.to_csv(output_dir / "same_hd_speed_pairs.csv", index=False)
speed_rates.to_csv(output_dir / "same_hd_speed_rates.csv")
display(hd_rates)
display(speed_rates)


In [ ]:
show(plot_matched_conditions(data, states, hd_pairs, hd_rates, unit_id, title="Similar HD, different positions"), f"unit_{unit_id}_same_hd")
show(plot_matched_conditions(data, speed_states, speed_pairs, speed_rates, unit_id, title="Similar HD and speed, different positions"), f"unit_{unit_id}_same_hd_speed")

## 3.3 HD-only simulations and correlations

For each of five contiguous test folds, estimate a 6° HD lookup on the other folds, excluding a 5 s guard. Circular smoothing is 3°. Draw 500 Poisson spike trains per unit along the real trajectory and apply the same occupancy normalization and EBC smoothing as the observed data. Compare band MRLs and spatial information with their null distributions. MRL p values are FDR-adjusted across all units and three bands. Shading in the unit plots is a pointwise interval.


In [ ]:
controls = hd_only_controls(data, simulations=simulations, workers=workers, seed=seed)
controls["table"].to_csv(output_dir / "hd_only_controls.csv")
display(controls["table"])


### Stability and sensitivity

Report Spearman correlation of the angular profiles between session halves, left/right spatial groups, all frames versus movement >2.5 cm/s, and the current smoothing versus σ=1 bin. Each correlation uses identical visited angle×distance support in its two conditions. These are descriptive correlations, without treating neighboring angle bins as independent samples.


In [ ]:
stability = stability_controls(data)
stability.to_csv(output_dir / "stability.csv")
evidence = evidence_table(controls, stability, geometry_rates, hd_rates)
hd = read_formatted_json(session / f"data/tuning_curves/Probe{probe}/tuning_curves.tc")
hd_classes = pd.Series(hd["unit_data"]["hd_class"], index=hd["unit_id"], dtype="Int64")
rf_ids = np.load(root / f"260831_1/data/rfmapping/good/-100_400_1ms/Probe{probe}/rf_sig_units.npy")
evidence["hd_class"] = hd_classes.reindex(evidence.index)
evidence["rf_significant"] = evidence.index.isin(rf_ids)
evidence = evidence.join(speed_rates.add_prefix("same_hd_speed_"))
evidence.to_csv(output_dir / "unit_evidence.csv")
display(evidence)
display(evidence.groupby(["conclusion", "rf_significant"]).size().rename("units"))
display(evidence.groupby(["hd_class", "conclusion"], dropna=False).size().unstack(fill_value=0))
show(plot_population_controls(evidence), "population_controls")


### Interpretation

“仍有额外 EBC 证据” requires HD-null MRL q<0.05 and split-half and across-position correlations ≥0.5 in the same distance band. Minimum support is 50 total spikes, 10 matched seconds per side in both comparisons, at least 20 occupancy-weighted spikes on each side, and 95% HD-prediction coverage. “采样不足” also marks significant but unstable effects; separate support/stability columns identify the reason. “HD/采样偏差可解释” means compatible with that explanation, not proof of its exclusivity. A fixed arena cannot by itself establish a causal EBC mechanism.

Spatial information uses the occupancy-weighted mean of the smoothed rate map. Zero-spike Poisson draws have undefined MRL and are omitted from that conditional null test; each band's defined null count is reported. The existing session-12 distance grid (0–28.99 cm) is retained for comparison with its saved maps; the geometry matches use all 60 untruncated ray distances.


In [ ]:
display(evidence.loc[[unit_id]])
show(plot_unit_controls(data, controls, unit_id), f"unit_{unit_id}_hd_only")
show(plot_unit_null_distributions(data, controls, unit_id), f"unit_{unit_id}_null_distributions")


In [ ]:
np.savez_compressed(output_dir / "control_arrays.npz", unit_ids=data["unit_ids"],
    observed=controls["observed"], predicted=controls["predicted"], curves=controls["curves"],
    expected_curves=controls["expected_curves"], null_curve_bounds=controls["null_curve_bounds"],
    null_strength=controls["null_strength"], null_spatial_information=controls["null_spatial_information"],
    occupancy=controls["occupancy"], distance_edges=data["edges"])
settings = {"session": str(session), "probe": probe, "phase": "baseline", "simulations": simulations,
            "bootstraps": bootstraps, "block_s": block_s, "seed": seed,
            "geometry_rms_cm": geometry_rms_cm, "position_separation_cm": position_separation_cm,
            "hd_separation_deg": hd_separation_deg, "position_bin_cm": 2, "hd_match_bin_deg": 12,
            "hd_folds": 5, "hd_guard_s": 5, "hd_smoothing_deg": 3,
            "ebc_sigma_bins": 5, "light_sigma_bins": 1, "speed_threshold_cm_s": 2.5}
(output_dir / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print("Results:", output_dir.resolve())

for name, item in (("behavior_states", states), ("speed_states", speed_states)):
    np.savez_compressed(output_dir / f"{name}.npz", unit_ids=data["unit_ids"],
        xy=item["xy"], hd=item["hd"], geometry=item["geometry"],
        occupancy=item["occupancy"], counts=item["counts"], keys=item["keys"])
